# NOTEBOOK 08 — TF-IDF & SIMILARITY

## Project
Job Market Analyzer for Career Readiness in South Africa

## Purpose
Calculate TF-IDF vectors and cosine similarity between candidate CV and job descriptions.

## Position in Pipeline

Previous: Notebook 07 — Resume Parser

Current: Notebook 08

Next: Notebook 09 — Job Matching

## Main Objective
Transform text into numerical vectors and calculate similarity scores between candidate and jobs.

## Input
- `data/processed/sa_it_jobs_features.csv` — job dataset
- `data/processed/candidate_profile.json` — candidate profile

## Output
- Similarity scores for all jobs
- Similarity distribution visualization

---

# NOTEBOOK ROADMAP

STEP 1 — Define Similarity Objective
STEP 2 — Load Candidate Representation
STEP 3 — Load Job Representation
STEP 4 — Prepare Text Inputs
STEP 5 — Fit TF-IDF Correctly
STEP 6 — Transform Candidate and Job Text
STEP 7 — Calculate Cosine Similarity
STEP 8 — Validate Similarity Output
STEP 9 — Interpret Similarity
STEP 10 — Save Similarity Features

---

# STEP 1 — Define Similarity Objective

## Purpose
Establish the similarity calculation goals.

## Objective
Calculate how similar the candidate's CV is to each job description using TF-IDF and cosine similarity.

## Formula
Cosine Similarity = (A · B) / (||A|| × ||B||)

## Method
Use sklearn's TfidfVectorizer and cosine_similarity.

---

# STEP 2 — Load Candidate Representation

## Purpose
Load the candidate profile for similarity calculation.

## Method
Load the candidate profile JSON.

In [ ]:
# Load candidate profile
import json

with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)

# Create candidate text representation
candidate_text = ' '.join(candidate['skills']) + ' ' + ' '.join(candidate['education']) + ' ' + ' '.join(candidate['projects'])

print(f"Candidate text length: {len(candidate_text)} characters")
print(f"Candidate skills: {candidate['skills']}")

---

# STEP 3 — Load Job Representation

## Purpose
Load the job dataset for similarity calculation.

## Method
Load the processed job dataset.

In [ ]:
# Load job dataset
df = pd.read_csv('data/processed/sa_it_jobs_features.csv')

print(f"Job dataset loaded: {df.shape[0]} jobs")
print(f"Columns: {list(df.columns)[:10]}...")

---

# STEP 4 — Prepare Text Inputs

## Purpose
Prepare the text data for TF-IDF vectorization.

## Method
Combine job description and skills into a single text field.

In [ ]:
# Prepare job text
df['job_text'] = df['Job_Description_Processed'].fillna('') + ' ' + df['Skills_Processed'].fillna('')

# Prepare candidate text
candidate_text = candidate_text.strip()

print(f"Job text prepared: {len(df)} jobs")
print(f"Candidate text: {len(candidate_text)} characters")

---

# STEP 5 — Fit TF-IDF Correctly

## Purpose
Fit TF-IDF vectorizer on job descriptions only (no data leakage).

## Method
Fit TfidfVectorizer on job text, then transform both job and candidate text.

## Data Science Decision
TF-IDF is fitted on job descriptions only to prevent data leakage from the candidate.

In [ ]:
# Fit TF-IDF on job descriptions only
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit on job text only
tfidf_vectorizer.fit(df['job_text'])

print(f"TF-IDF fitted on {len(df)} job descriptions")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

---

# STEP 6 — Transform Candidate and Job Text

## Purpose
Transform text into TF-IDF vectors.

## Method
Transform both candidate and job text using the fitted vectorizer.

In [ ]:
# Transform text
job_tfidf = tfidf_vectorizer.transform(df['job_text'])
candidate_tfidf = tfidf_vectorizer.transform([candidate_text])

print(f"Job TF-IDF matrix shape: {job_tfidf.shape}")
print(f"Candidate TF-IDF vector shape: {candidate_tfidf.shape}")

---

# STEP 7 — Calculate Cosine Similarity

## Purpose
Calculate similarity between candidate and each job.

## Method
Use sklearn's cosine_similarity function.

## Formula
cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)

In [ ]:
# Calculate cosine similarity
similarity_scores = cosine_similarity(candidate_tfidf, job_tfidf).flatten()

# Add to dataframe
df['tfidf_similarity'] = similarity_scores

# Normalize to 0-100 scale
df['tfidf_similarity_score'] = (df['tfidf_similarity'] * 100).round(2)

print("Similarity Score Statistics:")
print("="*60)
print(df['tfidf_similarity_score'].describe())

print(f"\nTop 5 Most Similar Jobs:")
top_5 = df.nlargest(5, 'tfidf_similarity_score')[['Job_Title', 'Company', 'Province', 'tfidf_similarity_score']]
print(top_5.to_string(index=False))

---

# STEP 8 — Validate Similarity Output

## Purpose
Verify similarity scores are reasonable.

## Method
Check score distribution and visualize.

In [ ]:
# Similarity distribution visualization
plt.figure(figsize=(12, 6))
sns.histplot(df['tfidf_similarity_score'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of TF-IDF Similarity Scores', fontsize=14, fontweight='bold')
plt.xlabel('Similarity Score (0-100)')
plt.ylabel('Number of Jobs')
plt.axvline(df['tfidf_similarity_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["tfidf_similarity_score"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/similarity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Mean similarity: {df['tfidf_similarity_score'].mean():.2f}")
print(f"Median similarity: {df['tfidf_similarity_score'].median():.2f}")
print(f"Std deviation: {df['tfidf_similarity_score'].std():.2f}")

---

# STEP 9 — Interpret Similarity

## Result Interpretation

### What happened?
TF-IDF vectors were created and cosine similarity was calculated between the candidate and all jobs.

### What does it mean?
The similarity scores indicate how closely the candidate's CV matches each job description in terms of text content.

### Why does it matter?
Text similarity is one component of job matching. It complements structured skill matching.

### What decision does it support?
The similarity scores will be used as a feature in the job matching algorithm.

### What limitation should be considered?
TF-IDF similarity is only one component of job matching. It does not capture semantic meaning or structured requirements.

---

# STEP 10 — Save Similarity Features

## Purpose
Save similarity scores for use in job matching.

## Method
Save the updated dataset with similarity scores.

In [ ]:
# Save similarity features
df.to_csv('data/processed/sa_it_jobs_with_similarity.csv', index=False)
print(f"Dataset with similarity saved to: data/processed/sa_it_jobs_with_similarity.csv")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")

# Save top similar jobs
top_similar = df.nlargest(20, 'tfidf_similarity_score')[['Job_ID', 'Job_Title', 'Company', 'Province', 'tfidf_similarity_score']]
top_similar.to_csv('outputs/top_similar_jobs.csv', index=False)
print(f"Top 20 similar jobs saved to: outputs/top_similar_jobs.csv")

---

# FINAL NOTEBOOK RESULTS

## What We Did
Calculated TF-IDF vectors and cosine similarity between candidate CV and job descriptions.

## Key Results
1. Fitted TF-IDF on job descriptions (no data leakage)
2. Calculated similarity scores for all jobs
3. Generated similarity distribution visualization
4. Identified top 20 most similar jobs

## Result Interpretation
The similarity scores provide a text-based matching component that complements structured skill matching.

## Data Science Decisions
- Fitted TF-IDF on job descriptions only to prevent data leakage
- Used unigrams and bigrams for better matching
- Normalized scores to 0-100 scale

## Outputs Produced
- Dataset with similarity: data/processed/sa_it_jobs_with_similarity.csv
- Top similar jobs: outputs/top_similar_jobs.csv
- Visualization: outputs/similarity_distribution.png

## Limitations
- TF-IDF does not capture semantic meaning
- Similarity is only one component of matching

## Connection to the Next Notebook
Notebook 09 will combine TF-IDF similarity with structured features to calculate the final job match score.